# Supplementation-Aware Split Distribution Check

This notebook complements the original split-distribution EDA instead of replacing it.

Use it when the small-dataset workflow may include supplementation from `extra_samples_for_small_dataset.json` and the `load_extra_samples_for_small_dataset_splits.py` loader.

It answers four practical questions:

1. What does the official exact FMA-small split look like?
2. What do the current processed `train_small`, `val_small`, and `test_small` parquet files contain right now?
3. What additions are described by the current supplementation payload and settings?
4. How would split sizes, genre counts, source mix, and split-drift metrics change after those additions are materialized?

This notebook is intentionally small-dataset specific. The existing notebook remains the generic split-shape sanity check.

In [1]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_workspace_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'MelCNN-MGR').exists() and (candidate / 'FMA').exists():
            return candidate
    raise FileNotFoundError('Could not locate workspace root containing MelCNN-MGR and FMA')


WORKSPACE_ROOT = find_workspace_root(Path.cwd())
TRACKS_CSV = WORKSPACE_ROOT / 'FMA' / 'fma_metadata' / 'tracks.csv'
PROCESSED_DIR = WORKSPACE_ROOT / 'MelCNN-MGR' / 'data' / 'processed'
SETTINGS_PATH = WORKSPACE_ROOT / 'MelCNN-MGR' / 'settings.json'
EXTRA_SAMPLES_PATH = PROCESSED_DIR / 'extra_samples_for_small_dataset.json'

TARGET_SUBSET = 'small'
SPLIT_ORDER = ['training', 'validation', 'test']
SPLIT_FILES = {
    'training': f'train_{TARGET_SUBSET}.parquet',
    'validation': f'val_{TARGET_SUBSET}.parquet',
    'test': f'test_{TARGET_SUBSET}.parquet',
}
TOP_N_GENRES = 12
BASE_RANDOM_SEED = 20260309

In [2]:
tracks = pd.read_csv(TRACKS_CSV, header=[0, 1], index_col=0)

official_mask = (
    (tracks[('set', 'subset')] == TARGET_SUBSET)
    & tracks[('track', 'genre_top')].notna()
    & tracks[('set', 'split')].isin(SPLIT_ORDER)
)
official_df = tracks.loc[official_mask, [('set', 'split'), ('track', 'genre_top')]].copy()
official_df.columns = ['split', 'genre_top']
official_df['source'] = 'fma'
official_df['sample_id'] = official_df.index.map(lambda track_id: f'fma:{int(track_id)}')
official_df.index.name = 'track_id'

processed_frames = []
for split_name, filename in SPLIT_FILES.items():
    split_path = PROCESSED_DIR / filename
    frame = pd.read_parquet(split_path).copy()
    frame['split'] = split_name
    frame.index.name = 'track_id'
    processed_frames.append(frame)
processed_df = pd.concat(processed_frames, axis=0)

settings_payload = json.loads(SETTINGS_PATH.read_text())
supplementation_settings = settings_payload['small_dataset_supplementation']
target_genres = supplementation_settings['target_genres']
n_extra_expected = int(supplementation_settings['n_extra_expected'])
train_ratio = float(supplementation_settings['train_n_val_test_split_ratio'])

extra_samples_payload = json.loads(EXTRA_SAMPLES_PATH.read_text())
selected_tracks = extra_samples_payload.get('selected_tracks', {})

print(f'Workspace root              : {WORKSPACE_ROOT}')
print(f'Official exact-small rows   : {len(official_df):,}')
print(f'Current processed split rows: {len(processed_df):,}')
print(f'Target genres in settings   : {target_genres}')
print(f'n_extra_expected per genre  : {n_extra_expected}')
print(f'train split ratio           : {train_ratio:.2f}')
print(f'Genres with selected payload: {sorted(selected_tracks)}')

Workspace root              : /mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1
Official exact-small rows   : 8,000
Current processed split rows: 8,000
Target genres in settings   : ['Hip-Hop', 'Pop', 'Folk', 'Rock', 'Electronic', 'Classical', 'Jazz', 'Country', 'Blues']
n_extra_expected per genre  : 300
train split ratio           : 0.80
Genres with selected payload: ['Blues', 'Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop', 'Jazz', 'Pop', 'Rock']


In [3]:
def stable_row_key(row: dict[str, object]) -> str:
    track_id = row.get('track_id')
    filepath = row.get('filepath') or row.get('relative_path') or row.get('filename') or ''
    return '|'.join([
        str(row.get('source') or ''),
        str(row.get('genre') or ''),
        '' if track_id is None else str(track_id),
        str(filepath),
    ])


def genre_seed(base_seed: int, genre: str) -> int:
    digest = hashlib.sha1(genre.encode('utf-8')).digest()
    genre_hash = int.from_bytes(digest[:4], byteorder='big', signed=False)
    return base_seed + genre_hash


def deterministic_shuffle(rows: list[dict[str, object]], seed: int, genre: str) -> list[dict[str, object]]:
    shuffled = list(rows)
    shuffled.sort(key=stable_row_key)
    random.Random(genre_seed(seed, genre)).shuffle(shuffled)
    return shuffled


def compute_split_counts(total: int, train_ratio: float) -> dict[str, int]:
    train_count = int(round(total * train_ratio))
    train_count = max(0, min(total, train_count))

    remainder = total - train_count
    if remainder % 2 == 1:
        train_count += 1
        remainder -= 1

    val_count = remainder // 2
    test_count = remainder // 2
    return {
        'training': train_count,
        'validation': val_count,
        'test': test_count,
    }


def counts_by_genre_and_split(frame: pd.DataFrame) -> pd.DataFrame:
    table = pd.crosstab(frame['genre_top'], frame['split'])
    return table.reindex(columns=SPLIT_ORDER, fill_value=0).sort_index()


def proportions_from_counts(counts: pd.DataFrame) -> pd.DataFrame:
    return counts.div(counts.sum(axis=0), axis=1).fillna(0.0)


def tvd(p: np.ndarray, q: np.ndarray) -> float:
    return 0.5 * np.abs(p - q).sum()


def jsd(p: np.ndarray, q: np.ndarray, eps: float = 1e-12) -> float:
    p = np.clip(p, eps, 1.0)
    p = p / p.sum()
    q = np.clip(q, eps, 1.0)
    q = q / q.sum()
    m = 0.5 * (p + q)
    return 0.5 * (np.sum(p * np.log2(p / m)) + np.sum(q * np.log2(q / m)))


def split_distance_metrics(counts: pd.DataFrame) -> dict[str, float]:
    props = proportions_from_counts(counts)
    genres = props.index
    p_train = props['training'].reindex(genres).to_numpy()
    p_val = props['validation'].reindex(genres).to_numpy()
    p_test = props['test'].reindex(genres).to_numpy()
    return {
        'TVD(train,val)': tvd(p_train, p_val),
        'TVD(train,test)': tvd(p_train, p_test),
        'JSD(train,val)': jsd(p_train, p_val),
        'JSD(train,test)': jsd(p_train, p_test),
    }

In [4]:
official_counts = counts_by_genre_and_split(official_df)
processed_counts = counts_by_genre_and_split(processed_df)

official_split_sizes = official_df['split'].value_counts().reindex(SPLIT_ORDER, fill_value=0)
processed_split_sizes = processed_df['split'].value_counts().reindex(SPLIT_ORDER, fill_value=0)
split_size_compare = pd.concat(
    [
        official_split_sizes.rename('official_exact_small'),
        processed_split_sizes.rename('processed_current'),
    ],
    axis=1,
)
split_size_compare['delta_vs_official'] = split_size_compare['processed_current'] - split_size_compare['official_exact_small']

processed_source_summary = (
    processed_df.groupby(['split', 'source']).size().unstack(fill_value=0).reindex(SPLIT_ORDER, fill_value=0)
)

current_has_non_fma_rows = 'source' in processed_df.columns and processed_df['source'].ne('fma').any()
new_genres_vs_official = sorted(set(processed_counts.index) - set(official_counts.index))

print('Current split-size comparison:')
display(split_size_compare)

print('Current processed source mix by split:')
display(processed_source_summary)

print(f'Non-FMA rows already present? {current_has_non_fma_rows}')
print(f'Genres outside official exact-small currently present: {new_genres_vs_official}')

print('Official exact-small genre counts:')
display(official_counts)

print('Current processed genre counts:')
display(processed_counts)

Current split-size comparison:


,official_exact_small,processed_current,delta_vs_official
split,,,
training,6400,6400,0
validation,800,800,0
test,800,800,0


Current processed source mix by split:


source,fma
split,
training,6400
validation,800
test,800


Non-FMA rows already present? False
Genres outside official exact-small currently present: []
Official exact-small genre counts:


split,training,validation,test
genre_top,,,
Electronic,800,100,100
Experimental,800,100,100
Folk,800,100,100
Hip-Hop,800,100,100
Instrumental,800,100,100
International,800,100,100
Pop,800,100,100
Rock,800,100,100


Current processed genre counts:


split,training,validation,test
genre_top,,,
Electronic,800,100,100
Experimental,800,100,100
Folk,800,100,100
Hip-Hop,800,100,100
Instrumental,800,100,100
International,800,100,100
Pop,800,100,100
Rock,800,100,100


In [5]:
allocation_records = []
allocation_summary_records = []

for genre in target_genres:
    genre_rows = list(selected_tracks.get(genre, []))[:n_extra_expected]
    shuffled_rows = deterministic_shuffle(genre_rows, BASE_RANDOM_SEED, genre)
    split_counts = compute_split_counts(len(shuffled_rows), train_ratio)

    start = 0
    for split_name in SPLIT_ORDER:
        stop = start + split_counts[split_name]
        assigned_rows = shuffled_rows[start:stop]
        start = stop

        for row in assigned_rows:
            allocation_records.append({
                'split': split_name,
                'genre_top': genre,
                'source': row.get('source', 'unknown'),
                'track_id': row.get('track_id'),
                'filepath': row.get('filepath'),
            })

    allocation_summary_records.append({
        'genre_top': genre,
        'selected_rows': len(shuffled_rows),
        'train_add': split_counts['training'],
        'val_add': split_counts['validation'],
        'test_add': split_counts['test'],
    })

allocation_df = pd.DataFrame(allocation_records)
allocation_summary = pd.DataFrame(allocation_summary_records).set_index('genre_top').sort_values('selected_rows', ascending=False)

if allocation_df.empty:
    projected_counts = processed_counts.copy()
    projected_source_summary = processed_source_summary.copy()
else:
    allocation_counts = counts_by_genre_and_split(allocation_df)
    projected_counts = processed_counts.add(allocation_counts, fill_value=0).astype(int)

    added_source_summary = (
        allocation_df.groupby(['split', 'source']).size().unstack(fill_value=0).reindex(SPLIT_ORDER, fill_value=0)
    )
    projected_source_summary = processed_source_summary.add(added_source_summary, fill_value=0).astype(int)

print('Projected additions per target genre:')
display(allocation_summary)

if allocation_df.empty:
    print('No selected supplementation rows were found in the payload.')
else:
    print('Projected added rows by split and source:')
    display(added_source_summary)

    print('Projected total source mix after materializing the payload:')
    display(projected_source_summary)

    projected_split_sizes = projected_counts.sum(axis=0).rename('projected_after_payload')
    size_compare = pd.concat([processed_split_sizes.rename('processed_current'), projected_split_sizes], axis=1)
    size_compare['delta_added'] = size_compare['projected_after_payload'] - size_compare['processed_current']
    print('Projected split sizes after materializing the payload:')
    display(size_compare)

Projected additions per target genre:


,selected_rows,train_add,val_add,test_add
genre_top,,,,
Hip-Hop,300,240,30,30
Pop,300,240,30,30
Folk,300,240,30,30
Rock,300,240,30,30
Electronic,300,240,30,30
Classical,300,240,30,30
Jazz,300,240,30,30
Country,300,240,30,30
Blues,294,236,29,29


Projected added rows by split and source:


source,dortmund,fma-medium,gtzan
split,,,
training,284,1790,82
validation,40,222,7
test,32,226,11


Projected total source mix after materializing the payload:


source,dortmund,fma,fma-medium,gtzan
split,,,,
training,284,6400,1790,82
validation,40,800,222,7
test,32,800,226,11


Projected split sizes after materializing the payload:


,processed_current,projected_after_payload,delta_added
split,,,
training,6400,8556,2156
validation,800,1069,269
test,800,1069,269


In [ ]:
current_split_sizes = processed_counts.sum(axis=0).reindex(SPLIT_ORDER)
projected_split_sizes = projected_counts.sum(axis=0).reindex(SPLIT_ORDER)

current_training = processed_counts['training'].sort_values(ascending=False)
projected_training = projected_counts['training'].sort_values(ascending=False)
plot_genres = (current_training.add(projected_training, fill_value=0)
               .sort_values(ascending=False)
               .head(TOP_N_GENRES)
               .index)
training_compare = pd.DataFrame({
    'current_training': current_training.reindex(plot_genres, fill_value=0),
    'projected_training': projected_training.reindex(plot_genres, fill_value=0),
})

fig, axes = plt.subplots(1, 3, figsize=(22, 5))

pd.DataFrame({
    'current': current_split_sizes,
    'projected': projected_split_sizes,
}).plot(kind='bar', ax=axes[0], width=0.75)
axes[0].set_title('Split sizes: current vs projected')
axes[0].set_ylabel('Rows')
axes[0].tick_params(axis='x', rotation=0)

training_compare.plot(kind='bar', ax=axes[1], width=0.8)
axes[1].set_title(f'Training split genre counts (top {TOP_N_GENRES})')
axes[1].set_ylabel('Rows')
axes[1].tick_params(axis='x', rotation=55)

projected_source_summary.plot(kind='bar', stacked=True, ax=axes[2], width=0.75)
axes[2].set_title('Projected source mix by split')
axes[2].set_ylabel('Rows')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
metrics_table = pd.DataFrame({
    'official_exact_small': split_distance_metrics(official_counts),
    'processed_current': split_distance_metrics(processed_counts),
    'projected_after_payload': split_distance_metrics(projected_counts),
}).T.round(4)

genre_delta = projected_counts.subtract(processed_counts, fill_value=0).astype(int)
genre_delta['total_added'] = genre_delta.sum(axis=1)
genre_delta = genre_delta.sort_values('total_added', ascending=False)

print('Split-distance metrics before and after materializing the current payload:')
display(metrics_table)

print('Per-genre added counts implied by the current payload:')
display(genre_delta[genre_delta['total_added'] > 0])

projected_props = proportions_from_counts(projected_counts)
fig, ax = plt.subplots(figsize=(14, 5))
projected_props.reindex(plot_genres).plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Projected genre proportions by split')
ax.set_ylabel('Proportion of rows')
ax.tick_params(axis='x', rotation=55)
plt.tight_layout()
plt.show()

## Interpretation Guide

Use the results in this order:

1. If the current processed splits match official exact-small and the source mix is all `fma`, supplementation has not been materialized into the split parquets yet.
2. The projected tables and plots show what the current JSON payload plus settings would do if passed through the split loader with the same deterministic allocation logic.
3. If validation and test receive added rows, the benchmark has changed. That is a valid experiment, but it should be documented as a supplemented benchmark variant rather than treated as the original FMA-small evaluation.
4. Non-FMA sources or genres outside the original 8 exact-small labels are the clearest signs that the live split parquets already contain supplemented rows.
5. Keep using the original notebook for generic split-shape sanity checks. Use this notebook when you need provenance-aware analysis of the small split workflow.